# Estudiante A (Carlos Orozco): VizieR

Campo profundo centrado en **RA = 135.5 grados**, **Dec = 0.5 grados**, con radio de **0.5 grados**. La consulta conserva Gaia DR3 como catalogo principal mediante un `LEFT JOIN` espacial con AllWISE a 2 segundos de arco.

## 1. Extraccion directa con Bash y ADQL

La consulta se escribe directamente y se envia al endpoint TAP con `wget`. No se utilizan librerias astronomicas.

In [ ]:
%%bash
echo "Esta consulta puede tardar varios minutos."
mkdir -p datos resultados

ADQL="SELECT g.Source, g.RA_ICRS, g.DE_ICRS, g.Plx, g.e_Plx, g.pmRA, g.pmDE, g.Gmag, w.AllWISE, w.W1mag FROM \"I/355/gaiadr3\" AS g LEFT JOIN \"II/328/allwise\" AS w ON 1=CONTAINS(POINT('ICRS', w.RAJ2000, w.DEJ2000), CIRCLE('ICRS', g.RA_ICRS, g.DE_ICRS, 2./3600.)) WHERE 1=CONTAINS(POINT('ICRS', g.RA_ICRS, g.DE_ICRS), CIRCLE('ICRS', 135.5, 0.5, 0.5))"
URL_ADQL=$(echo "$ADQL" | sed 's/ /+/g')
TAP_URL="https://tapvizier.cds.unistra.fr/TAPVizieR/tap/sync?request=doQuery&lang=ADQL&format=csv&query="
TAP_RESPALDO="https://tapvizier.u-strasbg.fr/TAPVizieR/tap/sync?request=doQuery&lang=ADQL&format=csv&query="

wget -q -O datos/gaia_allwise_crudo.csv "$TAP_URL$URL_ADQL"
if [ ! -s datos/gaia_allwise_crudo.csv ]; then
    echo "Servidor principal no disponible. Intentando el espejo de VizieR..."
    wget -q -O datos/gaia_allwise_crudo.csv "$TAP_RESPALDO$URL_ADQL"
fi
if [ ! -s datos/gaia_allwise_crudo.csv ]; then
    echo "No fue posible descargar los datos de VizieR."
    exit 1
fi
echo "Consulta terminada"


## 2. Limpieza y seleccion de estrellas galacticas

Se exige una paralaje positiva detectada con una relacion entre paralaje y error de al menos 3. Ademas, se requieren las magnitudes G y W1 para calcular el indice de color.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

datos = pd.read_csv('datos/gaia_allwise_crudo.csv')
print('Filas descargadas:', len(datos))
print('Fuentes Gaia diferentes:', datos['Source'].nunique())

columnas_numericas = ['Plx', 'e_Plx', 'pmRA', 'pmDE', 'Gmag', 'W1mag']

for columna in columnas_numericas:
    datos[columna] = pd.to_numeric(datos[columna], errors='coerce')

estrellas = datos.dropna(subset=['Plx', 'e_Plx', 'Gmag', 'W1mag']).copy()
estrellas = estrellas[(estrellas['Plx'] > 0) &
                      (estrellas['e_Plx'] > 0) &
                      (estrellas['Plx'] / estrellas['e_Plx'] >= 3)].copy()

estrellas = estrellas.drop_duplicates(subset='Source')
estrellas['G-W1'] = estrellas['Gmag'] - estrellas['W1mag']

print('Estrellas galacticas con fotometria completa:', len(estrellas))
estrellas.to_csv('resultados/gaia_allwise_limpio.csv', index=False)
estrellas[['Plx', 'pmRA', 'pmDE', 'Gmag', 'W1mag', 'G-W1']].describe()

## 3. Diagrama color-magnitud optico-infrarrojo

El color se define como `G-W1`. El eje de magnitud G se invierte siguiendo la convencion astronomica: los objetos mas brillantes tienen magnitudes menores.

In [ ]:
plt.figure(figsize=(8, 6))
plt.scatter(estrellas['G-W1'], estrellas['Gmag'], s=12, color='darkorange', alpha=0.65)
plt.gca().invert_yaxis()
plt.xlabel('Indice de color G - W1 (mag)')
plt.ylabel('Magnitud G de Gaia (mag)')
plt.title('Diagrama color-magnitud: estrellas del campo profundo')
plt.grid(alpha=0.25)
plt.tight_layout()
plt.savefig('resultados/diagrama_color_magnitud.png', dpi=150)
plt.show()

## Interpretacion

El cruce optico-infrarrojo compara la emision medida por Gaia con la emision a mayor longitud de onda medida por AllWISE. Una fuente con `G-W1` grande es relativamente mas brillante en infrarrojo que en el optico. Esto puede estar relacionado con una menor temperatura o con extincion por polvo, que atenúa mas la luz optica que la infrarroja.